# 01. Satellite Data Exploration & Multi-spectral Band Inspection
### Project: Image Change Detection using Digital Image Processing for Environmental Monitoring

This notebook demonstrates:
1. Loading multi-band Sentinel-2 L2A GeoTIFFs (B03 Green, B04 Red, B08 NIR, B11 SWIR1, B12 SWIR2) and plain RGB images.
2. Inspecting spatial georeferencing, affine transformation, and coordinate reference systems (CRS).
3. Visualizing individual spectral reflectance bands using Matplotlib.
4. Rendering **Natural Color (RGB)** and **False Color Infrared (NIR-R-G)** composite views.
5. Analyzing spectral reflectance distributions using multi-band histograms.


In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Connect to modular backend services
sys.path.append(str(Path.cwd().parent / "backend"))

from app.services.utils import load_image, generate_synthetic_pair, pixel_area_m2, normalize_to_uint8
from app.services.indices import USE_CASES


### Step 1: Ensure Dataset Availability
If real Copernicus Analytical GeoTIFF downloads are not present, generate realistic synthetic pairs for testing.


In [ ]:
project_root = Path.cwd().parent
use_case = "deforestation"

# Generate or verify synthetic dataset
pair_paths = generate_synthetic_pair(use_case=use_case, size=512, base_dir=str(project_root))
before_path = pair_paths["before_path"]
after_path = pair_paths["after_path"]
gt_path = pair_paths["ground_truth_path"]

print(f"Before Image Path: {before_path}")
print(f"After Image Path:  {after_path}")
print(f"Ground Truth Path: {gt_path}")


### Step 2: Load Multi-Spectral Imagery & Inspect Geospatial Metadata


In [ ]:
before_info = load_image(before_path)
after_info = load_image(after_path)

print("=" * 55)
print("SATELLITE SCENE METADATA")
print("=" * 55)
print(f"Array Shape (C, H, W): {before_info['shape']}")
print(f"Multispectral (>=4 bands): {before_info['is_multispectral']}")
print(f"Georeferenced: {before_info['is_georeferenced']}")
print(f"Coordinate Reference System (CRS): {before_info['crs']}")
print(f"Affine Transform:
{before_info['transform']}")

px_area = pixel_area_m2(before_info['transform'], before_info['crs'])
print(f"Ground Pixel Area: {px_area:.2f} m^2 (Equivalent to {np.sqrt(px_area):.1f}m spatial resolution)")


### Step 3: Inspect Individual Spectral Bands
Sentinel-2 Band Configuration:
- **Band 0**: B03 (Green - 560 nm)
- **Band 1**: B04 (Red - 665 nm)
- **Band 2**: B08 (Near-Infrared / NIR - 842 nm)
- **Band 3**: B11 (Short-Wave Infrared 1 / SWIR1 - 1610 nm)
- **Band 4**: B12 (Short-Wave Infrared 2 / SWIR2 - 2190 nm)


In [ ]:
band_labels = [
    "B03 (Green)", "B04 (Red)", "B08 (NIR)",
    "B11 (SWIR1)", "B12 (SWIR2)"
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i in range(5):
    im = axes[i].imshow(before_info["bands"][i], cmap="viridis")
    axes[i].set_title(band_labels[i], fontsize=12, fontweight="bold")
    axes[i].axis("off")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

plt.suptitle("Sentinel-2 Individual Spectral Bands (Surface Reflectance)", fontsize=15, y=1.05)
plt.tight_layout()
plt.show()


### Step 4: True Color vs False Color Infrared Composites
In remote sensing, **False Color Infrared (NIR -> Red, Red -> Green, Green -> Blue)** makes healthy chlorophyll-rich vegetation glow in bright red, making forest disturbance instantly detectable.


In [ ]:
bands = before_info["bands"]

# Natural Color: Red=B04 (index 1), Green=B03 (index 0), Blue=B03*0.85
natural_rgb = np.stack([
    normalize_to_uint8(bands[1]),
    normalize_to_uint8(bands[0]),
    normalize_to_uint8(bands[0] * 0.85)
], axis=-1)

# False Color Infrared: Red=B08 (NIR), Green=B04 (Red), Blue=B03 (Green)
false_color_ir = np.stack([
    normalize_to_uint8(bands[2]),
    normalize_to_uint8(bands[1]),
    normalize_to_uint8(bands[0])
], axis=-1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(natural_rgb)
axes[0].set_title("Natural Color Composite (RGB)", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(false_color_ir)
axes[1].set_title("False Color Infrared (NIR-Red-Green)
[Bright Red = Dense Canopy]", fontsize=13, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Step 5: Spectral Reflectance Histograms


In [ ]:
plt.figure(figsize=(10, 5))
colors = ["green", "red", "darkred", "purple", "brown"]

for i in range(5):
    plt.hist(
        before_info["bands"][i].ravel(),
        bins=60,
        range=(0.0, 0.8),
        color=colors[i],
        alpha=0.5,
        label=band_labels[i]
    )

plt.title("Spectral Reflectance Distribution per Band", fontsize=13, fontweight="bold")
plt.xlabel("Reflectance Value [0.0 - 1.0]", fontsize=11)
plt.ylabel("Pixel Frequency", fontsize=11)
plt.legend(frameon=True)
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()
